# Day 10: Adam: A Method for Stochastic Optimization (From-Scratch)

> **Bài báo gốc:** *Adam: A Method for Stochastic Optimization*  
> **Tác giả:** Diederik P. Kingma, Jimmy Lei Ba  
> **Xuất bản:** International Conference on Learning Representations (ICLR), 2015  
> **Tài liệu gốc:** [Day 10 Adam.pdf](./resources/Day%2010%20Adam.pdf)

---

## 📌 Giới Thiệu Chuyên Sâu Cho Người Mới Bắt Đầu

Trong quá trình huấn luyện mạng nơ-ron, mục tiêu của ta là tìm bộ trọng số $\theta$ sao cho hàm mất mát $\mathcal{L}(\theta)$ nhỏ nhất. Thuật toán tối ưu hóa (**Optimizer**) đóng vai trò như một người leo núi trong đêm sương mù, chỉ dựa vào độ dốc chân bước (**Gradient**) để tìm đường xuống thung lũng sâu nhất.

Trước khi **Adam** ra đời năm 2014:
1. **SGD:** Bước đi chậm chạp, dễ bị rung lắc dữ dội khi địa hình dốc lệch (*ravines*).
2. **Momentum:** Tích lũy vận tốc để vượt dốc nhưng dễ bị văng quán tính quá xa đích.
3. **AdaGrad:** Tự động điều chỉnh bước nhảy cho từng chiều, nhưng dồn tích bình phương gradient liên tục khiến bước đi nhanh chóng bị "đóng băng" về 0.
4. **RMSprop:** Khắc phục nhược điểm của AdaGrad bằng trung bình động hàm mũ, nhưng thiếu quán tính và thiếu hiệu chỉnh độ chệch ban đầu.

**Adam (Adaptive Moment Estimation)** ra đời như một sự kết hợp hoàn hảo:
- Tích hợp **Mô-men bậc 1** (Quán tính làm mượt hướng di chuyển).
- Tích hợp **Mô-men bậc 2** (Thích ứng bước nhảy theo từng tọa độ).
- Phát minh cơ chế **Hiệu Chỉnh Độ Chệch (Bias Correction)** giải tích độc đáo, giúp những bước đi đầu tiên không bị bùng nổ hay lệch lạc.


In [ ]:
import time
import numpy as np

print("[✓] Môi trường Python và NumPy đã sẵn sàng!")


## 1. Cài Đặt Các Bộ Tối Ưu Hóa Tiền Bối (SGD, Momentum, AdaGrad, RMSprop)

Để thấy rõ bước nhảy vọt của Adam, ta xây dựng lại các giải thuật kinh điển theo cùng một giao diện:
- **SGD:** $\theta_t = \theta_{t-1} - \alpha g_t$
- **Momentum:** $v_t = \gamma v_{t-1} + \alpha g_t, \quad \theta_t = \theta_{t-1} - v_t$
- **AdaGrad:** $G_t = G_{t-1} + g_t^2, \quad \theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{G_t} + \epsilon} g_t$
- **RMSprop:** $v_t = \beta v_{t-1} + (1 - \beta) g_t^2, \quad \theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{v_t} + \epsilon} g_t$


In [ ]:
class BaseOptimizer:
    def __init__(self, lr=0.01):
        self.lr = lr
        self.t = 0

    def step(self, params, grads):
        raise NotImplementedError


class SGDOptimizer(BaseOptimizer):
    def __init__(self, lr=0.01, momentum=0.0):
        super().__init__(lr=lr)
        self.momentum = momentum
        self.v = {}

    def step(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.v:
                self.v[k] = np.zeros_like(params[k])
            if self.momentum > 0.0:
                self.v[k] = self.momentum * self.v[k] + self.lr * grads[k]
                params[k] -= self.v[k]
            else:
                params[k] -= self.lr * grads[k]


class AdaGradOptimizer(BaseOptimizer):
    def __init__(self, lr=0.1, eps=1e-8):
        super().__init__(lr=lr)
        self.eps = eps
        self.G = {}

    def step(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.G:
                self.G[k] = np.zeros_like(params[k])
            self.G[k] += grads[k] ** 2
            step_size = self.lr / (np.sqrt(self.G[k]) + self.eps)
            params[k] -= step_size * grads[k]


class RMSpropOptimizer(BaseOptimizer):
    def __init__(self, lr=0.01, beta=0.9, eps=1e-8):
        super().__init__(lr=lr)
        self.beta = beta
        self.eps = eps
        self.v = {}

    def step(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.v:
                self.v[k] = np.zeros_like(params[k])
            self.v[k] = self.beta * self.v[k] + (1.0 - self.beta) * (grads[k] ** 2)
            step_size = self.lr / (np.sqrt(self.v[k]) + self.eps)
            params[k] -= step_size * grads[k]

print("[✓] Đã cài đặt xong SGD, Momentum, AdaGrad, RMSprop!")


## 2. Đột Phá Kỹ Thuật Của Adam: Mô-men & Hiệu Chỉnh Độ Chệch

Trong Adam, hai thống kê chuyển động hàm mũ được duy trì:
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

### Tại sao bắt buộc phải có Hiệu Chỉnh Độ Chệch (Bias Correction)?
Vì khởi tạo $m_0 = 0, v_0 = 0$, nên ở những bước đầu:
$$\mathbb{E}[m_t] = \mathbb{E}[g_t] \cdot (1 - \beta_1^t)$$
$$\mathbb{E}[v_t] = \mathbb{E}[g_t^2] \cdot (1 - \beta_2^t)$$

Với $\beta_2 = 0.999$, tại $t=1$, $1 - \beta_2^1 = 0.001$. Giá trị $v_1$ bị thu nhỏ gấp 1000 lần so với thực tế! Nếu không chia cho $(1 - \beta_2^t)$, mẫu số $\sqrt{v_1}$ cực kỳ bé sẽ làm bước nhảy $\frac{\alpha}{\sqrt{v_1}} m_1$ bùng nổ làm phá hủy mạng nơ-ron ngay lượt lặp đầu tiên.

Do đó, công thức ước lượng không chệch là:
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$


In [ ]:
class AdamOptimizer(BaseOptimizer):
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, bias_correction=True):
        super().__init__(lr=lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.bias_correction = bias_correction
        self.m = {}
        self.v = {}

    def step(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.m:
                self.m[k] = np.zeros_like(params[k])
                self.v[k] = np.zeros_like(params[k])

            # Cập nhật mô-men bậc 1 & 2
            self.m[k] = self.beta1 * self.m[k] + (1.0 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1.0 - self.beta2) * (grads[k] ** 2)

            if self.bias_correction:
                m_hat = self.m[k] / (1.0 - (self.beta1 ** self.t))
                v_hat = self.v[k] / (1.0 - (self.beta2 ** self.t))
            else:
                m_hat = self.m[k]
                v_hat = self.v[k]

            step_size = self.lr / (np.sqrt(v_hat) + self.eps)
            params[k] -= step_size * m_hat


class AdamWOptimizer(BaseOptimizer):
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        super().__init__(lr=lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = {}
        self.v = {}

    def step(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.m:
                self.m[k] = np.zeros_like(params[k])
                self.v[k] = np.zeros_like(params[k])

            # Decoupled Weight Decay: phân rã trọng số trực tiếp
            if self.weight_decay > 0.0:
                params[k] -= self.lr * self.weight_decay * params[k]

            self.m[k] = self.beta1 * self.m[k] + (1.0 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1.0 - self.beta2) * (grads[k] ** 2)

            m_hat = self.m[k] / (1.0 - (self.beta1 ** self.t))
            v_hat = self.v[k] / (1.0 - (self.beta2 ** self.t))

            params[k] -= (self.lr / (np.sqrt(v_hat) + self.eps)) * m_hat

print("[✓] Đã cài đặt xong Adam (ICLR 2015) và AdamW (ICLR 2019)!")


## 3. Thực Nghiệm 1: Thử Thách Địa Hình Hẻm Núi Dốc Lệch (Ravine)

Ta kiểm chứng 5 thuật toán trên hàm mất mát có độ cong dị hướng (Anisotropic):
$$f(x, y) = 0.5 x^2 + 25.0 y^2$$
- Trục $x$: Đáy thung lũng thoai thoải (Curvature = 1.0).
- Trục $y$: Vách hẻm núi dốc đứng gấp 50 lần (Curvature = 50.0).
- Điểm xuất phát: $(-4.0, 1.0)$. Cực tiểu toàn cục tại $(0.0, 0.0)$.


In [ ]:
def ravine_loss_and_grad(x, y, curvature_ratio=50.0):
    loss = 0.5 * (x ** 2) + 0.5 * curvature_ratio * (y ** 2)
    grad_x = x
    grad_y = curvature_ratio * y
    return loss, {"x": grad_x, "y": grad_y}

opts = {
    "1. Vanilla SGD": SGDOptimizer(lr=0.035, momentum=0.0),
    "2. SGD + Momentum": SGDOptimizer(lr=0.02, momentum=0.85),
    "3. AdaGrad": AdaGradOptimizer(lr=1.2),
    "4. RMSprop": RMSpropOptimizer(lr=0.08, beta=0.9),
    "5. Adam (2014)": AdamOptimizer(lr=0.15, beta1=0.9, beta2=0.999),
}

print(f"{'Thuật Toán':<20} | {'Loss Cuối':<14} | {'Vị trí Cuối (x, y)':<20} | {'Chiều Dài Quỹ Đạo'}")
print("-" * 75)

for name, opt in opts.items():
    p = {"x": np.array(-4.0), "y": np.array(1.0)}
    path_len = 0.0
    for _ in range(200):
        loss, g = ravine_loss_and_grad(p["x"], p["y"])
        prev = np.array([p["x"], p["y"]])
        opt.step(p, g)
        curr = np.array([p["x"], p["y"]])
        path_len += np.linalg.norm(curr - prev)

    pos = f"({p['x']:.2f}, {p['y']:.2f})"
    print(f"{name:<20} | {loss:<14.2e} | {pos:<20} | {path_len:6.2f}")


## 4. Thực Nghiệm 2: Huấn Luyện Mạng Nơ-ron Đa Tầng (MLP)

Ta huấn luyện một mạng nơ-ron gồm 2 tầng ẩn trên tập dữ liệu nhị phân với đặc trưng thưa (Sparse features). 
Mục tiêu: Đánh giá tốc độ suy giảm hàm mất mát qua 100 epochs giữa các thuật toán tối ưu hóa.


In [ ]:
class SimpleMLP:
    def __init__(self, in_features=20, hidden_dim=32, out_features=2, seed=42):
        np.random.seed(seed)
        self.params = {
            "W1": np.random.randn(in_features, hidden_dim) * np.sqrt(2.0 / in_features),
            "b1": np.zeros((1, hidden_dim)),
            "W2": np.random.randn(hidden_dim, out_features) * np.sqrt(2.0 / hidden_dim),
            "b2": np.zeros((1, out_features)),
        }
        self.cache = {}

    def forward(self, X):
        self.cache["X"] = X
        z1 = np.dot(X, self.params["W1"]) + self.params["b1"]
        a1 = np.maximum(0.0, z1)
        self.cache["z1"] = z1
        self.cache["a1"] = a1
        logits = np.dot(a1, self.params["W2"]) + self.params["b2"]
        exp_s = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exp_s / (np.sum(exp_s, axis=1, keepdims=True) + 1e-12)
        self.cache["probs"] = probs
        return probs

    def backward(self, y_true):
        probs, X, a1, z1 = self.cache["probs"], self.cache["X"], self.cache["a1"], self.cache["z1"]
        N = X.shape[0]
        loss = np.mean(-np.log(probs[np.arange(N), y_true] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(N), y_true] -= 1.0
        dlogits /= N
        dW2 = np.dot(a1.T, dlogits)
        db2 = np.sum(dlogits, axis=0, keepdims=True)
        da1 = np.dot(dlogits, self.params["W2"].T)
        dz1 = da1 * (z1 > 0).astype(np.float32)
        dW1 = np.dot(X.T, dz1)
        db1 = np.sum(dz1, axis=0, keepdims=True)
        return loss, {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

# Sinh dữ liệu
np.random.seed(1337)
N, D = 300, 20
X = np.random.randn(N, D)
X[np.abs(X) < 1.0] = 0.0 # Thưa 70%
y = ((X[:, 0] * 2.0 - X[:, 1] * 1.5 + X[:, 2] * 0.5) > 0).astype(np.int64)

benchmark_opts = {
    "Vanilla SGD": SGDOptimizer(lr=0.02),
    "SGD + Momentum": SGDOptimizer(lr=0.02, momentum=0.9),
    "AdaGrad": AdaGradOptimizer(lr=0.08),
    "RMSprop": RMSpropOptimizer(lr=0.02, beta=0.9),
    "Adam (2014)": AdamOptimizer(lr=0.02, beta1=0.9, beta2=0.999),
    "AdamW (2017)": AdamWOptimizer(lr=0.02, weight_decay=0.01),
}

print(f"{'Thuật Toán':<18} | {'Loss Ep 0':<10} | {'Loss Ep 50':<12} | {'Loss Cuối':<10} | {'Train Acc'}")
print("-" * 70)

for name, opt in benchmark_opts.items():
    mlp = SimpleMLP(in_features=D, hidden_dim=32, out_features=2, seed=42)
    losses = []
    for ep in range(100):
        mlp.forward(X)
        loss, grads = mlp.backward(y)
        opt.step(mlp.params, grads)
        losses.append(loss)
    preds = np.argmax(mlp.forward(X), axis=1)
    acc = np.mean(preds == y) * 100.0
    print(f"{name:<18} | {losses[0]:8.4f}  | {losses[49]:8.4f}    | {losses[-1]:8.4f}  | {acc:6.1f}%")


## 5. Tổng Kết & Bài Học Cho Người Mới Bắt Đầu

1. **Hiểu bản chất của Adam:** Adam = Momentum (làm mượt hướng di chuyển) + RMSprop (tự động co giãn bước nhảy từng chiều) + Bias Correction (chống nổ bước nhảy ở epoch đầu).
2. **Quy tắc chọn Learning Rate:** Với Adam, giá trị khởi đầu phổ biến nhất là $\alpha = 0.001$ ($10^{-3}$) hoặc $0.0003$ ($3 \times 10^{-4}$ - "Karpathy constant").
3. **Khi nào nên dùng AdamW thay cho Adam:** Khi mạng nơ-ron của bạn sử dụng Weight Decay ($L_2$ regularization), đặc biệt với kiến trúc **Transformers**, luôn ưu tiên **AdamW** để tránh việc trọng số có gradient lớn bị suy giảm khả năng điều hòa!
